In [1]:
!nvidia-smi

Mon Mar 16 17:40:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:01:00.0  On |                  Off |
|  0%   30C    P8             14W /  450W |    7612MiB /  24564MiB |      8%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install numba
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from numba import cuda
import numpy as np
print(cuda.gpus)

# Define the cuda kernel
@cuda.jit
def add_kernel(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[idx] = a[idx] + b[idx]

# Host Code (CPU)
n = 1000000
a_host = np.random.randn(n).astype(np.float32)
b_host = np.random.randn(n).astype(np.float32)
c_host = np.empty_like(a_host)

# Allocate memory on the GPU
a_device = cuda.to_device(a_host)
b_device = cuda.to_device(b_host)
c_device = cuda.device_array_like(c_host)

# Configure the kernel launch parameters of the grid and block
threads_per_block = 256
# Calculate how many blocks we need to cover the entire array
blocks_per_grid = (a_device.size + (threads_per_block - 1)) // threads_per_block
print(f"Launching {blocks_per_grid} blocks of {threads_per_block} threads each")

# Launch the kernel on the gpu
add_kernel[blocks_per_grid, threads_per_block](a_device, b_device, c_device)

# Copy the result back to the host
c_device.copy_to_host(c_host)

if np.allclose(c_host, a_host + b_host):
    print("GPU results match CPU results!")
else:
    print("GPU results do not match CPU results!")
    
    


<Managed Device 0>
Launching 3907 blocks of 256 threads each
GPU results match CPU results!


Modify the code so that instead of writing to c[idx], you write to c[0]

In [3]:
from numba import cuda
import numpy as np
print(cuda.gpus)

# Define the cuda kernel
@cuda.jit
def add_kernel(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[0] = a[idx] + b[idx]

# Host Code (CPU)
n = 1000000
a_host = np.random.randn(n).astype(np.float32)
b_host = np.random.randn(n).astype(np.float32)
c_host = np.empty_like(a_host)

# Allocate memory on the GPU
a_device = cuda.to_device(a_host)
b_device = cuda.to_device(b_host)
c_device = cuda.device_array_like(c_host)

# Configure the kernel launch parameters of the grid and block
threads_per_block = 256
# Calculate how many blocks we need to cover the entire array
blocks_per_grid = (a_device.size + (threads_per_block - 1)) // threads_per_block
print(f"Launching {blocks_per_grid} blocks of {threads_per_block} threads each")

# Launch the kernel on the gpu
add_kernel[blocks_per_grid, threads_per_block](a_device, b_device, c_device)

# Copy the result back to the host
c_device.copy_to_host(c_host)

if np.allclose(c_host, a_host + b_host):
    print("GPU results match CPU results!")
else:
    print("GPU results do not match CPU results!")

print("c_host[0]: ", c_host[0])
print("c_host[1]: ", c_host[1])
print("a_host[-1] + b_host[-1]: ", a_host[-1] + b_host[-1])

<Managed Device 0>
Launching 3907 blocks of 256 threads each
GPU results do not match CPU results!
c_host[0]:  -0.6513994
c_host[1]:  0.0
a_host[-1] + b_host[-1]:  1.4892063


c_host[0] is some number equal to a_host[-1] + b_host[-1] because it rewrites the same exact index in every thread.

In [ ]:
from numba import cuda
import numpy as np
print(cuda.gpus)

# Define the cuda kernel
@cuda.jit
def add_kernel(a, b, c):
    idx = cuda.grid(1)
    # if idx < a.size:
    c[idx] = a[idx] + b[idx]

# Host Code (CPU)
n = 10000000
a_host = np.random.randn(n).astype(np.float32)
b_host = np.random.randn(n).astype(np.float32)
c_host = np.empty_like(a_host)

# Allocate memory on the GPU
a_device = cuda.to_device(a_host)
b_device = cuda.to_device(b_host)
c_device = cuda.device_array_like(c_host)

# Configure the kernel launch parameters of the grid and block
threads_per_block = 256
# Calculate how many blocks we need to cover the entire array
blocks_per_grid = (a_device.size + (threads_per_block - 1)) // threads_per_block
print(f"Launching {blocks_per_grid} blocks of {threads_per_block} threads each")

# Launch the kernel on the gpu
add_kernel[blocks_per_grid, threads_per_block](a_device, b_device, c_device)

# Copy the result back to the host
c_device.copy_to_host(c_host)

if np.allclose(c_host, a_host + b_host):
    print("GPU results match CPU results!")
else:
    print("GPU results do not match CPU results!")
    
    


<Managed Device 0>
Launching 3907 blocks of 256 threads each
GPU results match CPU results!
